# Data Processing with pandas

pandas is the standard library for working with tabular data in Python. A DataFrame is like a spreadsheet you can manipulate with code: filter rows, compute new columns, group and aggregate, and join tables, all in a few lines. If you've ever found yourself doing repetitive data work in Excel, pandas is the answer.

**What's inside:** creating DataFrames, inspecting data, selecting rows and columns with `loc`/`iloc`, boolean filtering, `apply()`, handling missing data, `groupby` aggregation, sorting, and merging.

**Learn more:** [pandas documentation](https://pandas.pydata.org/)

## Setup

In [ ]:
%pip install pandas

## 1. Creating DataFrames

### 1.1 From a dictionary

Each key becomes a column name; each list becomes the column values.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':   [24, 31, 22, 28, 35],
    'score': [88.5, 74.0, 95.0, 61.5, 82.0],
    'grade': ['B', 'C', 'A', 'D', 'B'],
    'city':  ['Austin', 'Boston', 'Austin', 'Denver', 'Boston'],
})
df

### 1.2 From a list of dicts

Each dict is one row. Missing keys become NaN.

In [ ]:
# each dict is one row
rows = [{'item': 'apple', 'qty': 3}, {'item': 'banana', 'qty': 5}]
pd.DataFrame(rows)

### 1.3 From a CSV string

Useful for quick inline test data without a file.

In [ ]:
from io import StringIO

csv = 'x,y\n1,10\n2,20\n3,30'
pd.read_csv(StringIO(csv))

## 2. Inspecting DataFrames

### 2.1 head / tail

In [ ]:
df.head(3)

In [ ]:
df.tail(2)

### 2.2 shape and dtypes

In [ ]:
df.shape   # (rows, columns)

In [ ]:
df.dtypes

### 2.3 info

Shows column names, non-null counts, and dtypes in one call.

In [ ]:
df.info()

### 2.4 describe

Summary statistics for all numeric columns.

In [ ]:
df.describe()

## 3. Selecting Data

### 3.1 Single column

In [ ]:
df['score']

### 3.2 Multiple columns

In [ ]:
df[['name', 'score']]

### 3.3 loc: label-based selection

`loc[row_label, column_name]`: rows and columns by their names/index labels.

In [ ]:
# single value by row label and column name
df.loc[2, 'name']

In [ ]:
# rows 1 through 3, selected columns
df.loc[1:3, ['name', 'score']]

### 3.4 iloc: position-based selection

`iloc[row_index, col_index]`: rows and columns by integer position.

In [ ]:
# last two rows, first two columns
df.iloc[-2:, :2]

## 4. Boolean Filtering

### 4.1 Single condition

In [ ]:
df[df['score'] > 80]

### 4.2 Multiple conditions

Use `&` for AND, `|` for OR; each condition must be wrapped in parentheses.

In [ ]:
df[(df['score'] > 70) & (df['city'] == 'Boston')]

### 4.3 isin

Filter rows where a column's value is in a set.

In [ ]:
df[df['grade'].isin(['A', 'B'])]

### 4.4 query()

Write filter conditions as a readable string, especially useful for complex filters.

In [ ]:
df.query('score > 80 and city == "Austin"')

## 5. Adding and Modifying Columns

### 5.1 New column from arithmetic

In [ ]:
df['score_pct'] = df['score'] / 100
df[['name', 'score', 'score_pct']].head(3)

### 5.2 apply() with a lambda

In [ ]:
# classify each score as pass or fail
df['result'] = df['score'].apply(lambda x: 'pass' if x >= 70 else 'fail')
df[['name', 'score', 'result']]

### 5.3 apply() with a named function

In [ ]:
def age_group(age):
    return 'young' if age < 30 else 'senior'

df['age_group'] = df['age'].apply(age_group)
df[['name', 'age', 'age_group']]

### 5.4 apply() across rows with axis=1

`axis=1` passes each row as a Series, letting you access multiple columns per row.

In [ ]:
df['label'] = df.apply(lambda r: r['name'] + ' (' + r['city'] + ')', axis=1)
df['label']

## 6. Handling Missing Data

In [ ]:
messy = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol'],
    'score': [88.5, None, 95.0],
    'city':  ['Austin', 'Boston', None],
})
messy

### 6.1 Detecting nulls

In [ ]:
messy.isna().sum()   # null count per column

### 6.2 dropna

In [ ]:
messy.dropna()   # drop any row with at least one null

In [ ]:
messy.dropna(subset=['score'])   # only drop rows where 'score' is null

### 6.3 fillna

In [ ]:
messy.fillna({'score': 0, 'city': 'Unknown'})

## 7. groupby and Aggregation

### 7.1 Single aggregation

In [ ]:
df.groupby('city')['score'].mean()

### 7.2 Multiple aggregations with agg()

In [ ]:
df.groupby('city')['score'].agg(['mean', 'max', 'count'])

### 7.3 Multiple columns

In [ ]:
df.groupby('city')[['age', 'score']].mean()

## 8. Sorting

In [ ]:
df.sort_values('score', ascending=False)[['name', 'score']]

In [ ]:
# sort by multiple columns (city ascending, score descending within each city)
df.sort_values(['city', 'score'], ascending=[True, False])[['name', 'city', 'score']]

## 9. Merging DataFrames

In [ ]:
# a second table referencing student names
sales = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Alice', 'Carol'],
    'amount': [200, 150, 300, 100],
})

### 9.1 Inner join (default)

Keep only rows with matching keys in both tables.

In [ ]:
pd.merge(df[['name', 'city']], sales, on='name')

### 9.2 Left join

Keep all rows from the left table; NaN where the right table has no match.

In [ ]:
pd.merge(df[['name', 'score']], sales, on='name', how='left')